## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [ ]:
import subprocess
result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [1]:
from pathlib import Path
import subprocess
import time

input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles")

cmd = [
    "tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

print(result.stderr)   # tippecanoe writes progress to stderr
print(f"\nCompleted in {elapsed:.1f}s")

FileNotFoundError: [Errno 2] No such file or directory: 'tippecanoe'

## Inspecting the Output

In [2]:
size_mb = output_file.stat().st_size / 1_000_000
raw_mb  = input_file.stat().st_size  / 1_000_000

print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")
print(f"Output (PMTiles):      {size_mb:.2f} MB")
print(f"Compression ratio:     {raw_mb / size_mb:.1f}×  smaller")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/railroads.pmtiles'

## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [3]:
# Use tippecanoe's companion tool to show metadata
result = subprocess.run(
    ["tile-join", "--no-tile-compression", "--if-matched",
     f"--output={output_file.with_suffix('.inspect.pmtiles')}",
     str(output_file)],
    capture_output=True, text=True
)

# Simpler: just show what pmtiles show gives us
result2 = subprocess.run(
    ["pmtiles", "show", str(output_file)],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr or "(install pmtiles CLI: pip install pmtiles)")

FileNotFoundError: [Errno 2] No such file or directory: 'tile-join'

## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [4]:
# Try loading with localtileserver if available
try:
    from localtileserver import TileClient, get_leaflet_tile_layer
    from ipyleaflet import Map

    client = TileClient(str(output_file))
    layer  = get_leaflet_tile_layer(client)
    m = Map(center=client.center(), zoom=client.default_zoom)
    m.add(layer)
    m
except ImportError:
    print("localtileserver not installed.")
    print("Install with: pip install localtileserver")
    print()
    print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
    print(f"File location: {output_file.resolve()}")

localtileserver not installed.
Install with: pip install localtileserver

Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.
File location: /Users/carlosastudillo/Spatial data 4543-5993 Astudillo/4545-5993-Spacial-Data/Assignments/03-Data_Manager/data/railroads.pmtiles


## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [5]:
# Run tippecanoe with --maximum-zoom=8 and compare output size
# Your code here
# Modify the command to set maximum zoom to 8
cmd[cmd.index('--maximum-zoom=14')] = '--maximum-zoom=8'

# Run the modified command and capture the output
t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

# Print the output and elapsed time
print(result.stderr)   # tippecanoe writes progress to stderr
print(f"\nCompleted in {elapsed:.1f}s")

# Compare output file size
size_mb = output_file.stat().st_size / 1_000_000
print(f"Output (PMTiles) with maximum zoom 8: {size_mb:.2f} MB")

FileNotFoundError: [Errno 2] No such file or directory: 'tippecanoe'

## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [6]:
# Calculate simplification tolerance in meters at different zoom levels
# Compare to our Module 02 epsilon choices
# Your code here
# BEGIN:
zoom_levels = [2, 5, 8, 12]
tile_size_km = 2.4  # size of a tile at zoom level 14 in km
pixels_per_tile = 4096

# Calculate the simplification tolerance in meters for each zoom level
tolerances_m = {}
for zoom in zoom_levels:
    scale_factor = 2 ** (14 - zoom)  # scale factor based on zoom level
    tolerance_m = (10 / pixels_per_tile) * tile_size_km * 1000 * scale_factor  # convert to meters
    tolerances_m[zoom] = tolerance_m

# Print the results
for zoom, tolerance in tolerances_m.items():
    print(f"Zoom level {zoom}: Simplification tolerance = {tolerance:.2f} meters")
# END:

Zoom level 2: Simplification tolerance = 24000.00 meters
Zoom level 5: Simplification tolerance = 3000.00 meters
Zoom level 8: Simplification tolerance = 375.00 meters
Zoom level 12: Simplification tolerance = 23.44 meters


## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

---
Tippecanoe drops by local spatial density (automatic but context-blind), while scalerank drops by global human-assigned importance (principled but can blank out entire regions) — neither is perfect, and a real pipeline would combine both.

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?